# LMM Analysis: Group × Level
## Metrics: LSCC, Mean Degree, ASPL (global)

---

## Method

### Participants

119 French-speaking adults: **51 SSD participants** (Group 100) and **68 controls** (Group 300).

### Material and Design

Each participant produced **9 verbal descriptions** of picture sequences, organised into **3 difficulty levels** (3 stories per level):

- Level 1: Bottes, Épicerie, MiseB
- Level 2: Camping, Canoë, Jongle
- Level 3: CerfVo, Chauss, Plage

**Group** is a between-subjects factor (SSD vs. Control); **Level** is a within-subjects factor with 3 modalities.

### Metrics

| Abbrev. | Full name |
|---------|----------|
| LSCC | Largest strongly connected component size |
| AD | Mean degree (edges / nodes) |
| ASPL | Average shortest path length (within LSCC) |

### Statistical Models (LMM)

Estimated by **REML** (final model) and **ML** (model comparisons).

**Fixed effects:**
- `group_ssd` (0 = Control, 1 = SSD)
- `niveau_lin` (orthogonal linear contrast: {1: −1, 2: 0, 3: +1})
- `group_ssd × niveau_lin` (linear interaction)
- `nb_tokens_z` (z-scored token count — included or excluded depending on the model)

**Random effects:**
- Random intercept per participant
- Random slope for `niveau_lin` per participant (selected by LRT)
- Cross-classified variance component per story (BD)

**Interaction test:** LRT χ²(1) — full model vs. model without the interaction term (ML) — **using the same random-effects structure**.

**Multiple comparisons correction:** Benjamini-Hochberg FDR (q = 0.05); Holm-Bonferroni also reported.

## Data

In [ ]:
# Standard library
import warnings

# Data & numerics
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import chi2 as chi2_dist, shapiro
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('Setup OK')

In [ ]:
# Graph metrics to extract and analyse
TARGET_G = ['sg_aspl', 'sg_avg_degree', 'sg_lscc_size', 'sg_lscc_ratio', 'sg_tokens']

df_raw = pd.read_csv('../SpeechGraph/data_SpeechGraph2.csv', sep=';')
df_raw.columns = df_raw.columns.str.strip()

# Keep groups 100 (SSD) and 300 (Control); drop rows with zero tokens
df_v = df_raw[
    df_raw['Groupe'].isin([100, 300])
    & (df_raw['n_tokens'] > 0)
].copy()
df_v['Groupe']     = df_v['Groupe'].astype(int)
df_v['No anonyme'] = df_v['No anonyme'].astype(int)

# Rename precomputed columns to internal naming convention
df_v = df_v.rename(columns={
    'LSCC':     'sg_lscc_size',
    'AD':       'sg_avg_degree',
    'ASPL':     'sg_aspl',
    'n_tokens': 'sg_tokens',
})
df_v['sg_lscc_ratio'] = df_v['sg_lscc_size'] / df_v['n_nodes'].replace(0, float('nan'))
df_v['nb_tokens'] = df_v['sg_tokens']

df_metrics = df_v[['No anonyme', 'Groupe', 'BD', 'Niveau', 'nb_tokens'] + TARGET_G].copy()

print(f'Verbatims: {len(df_metrics)}')
print(f'  Group 100 (SSD)    : {(df_metrics.Groupe==100).sum()}')
print(f'  Group 300 (Control): {(df_metrics.Groupe==300).sum()}')
print(f'  Stories (BD)       : {df_metrics.BD.nunique()}')
print(f'  Levels             : {sorted(df_metrics.Niveau.unique())}')
print('\nDistribution BD × Level:')
print(df_metrics.groupby(['BD', 'Niveau']).size().unstack(fill_value=0))

In [ ]:
# Sanity check: missing values and overall descriptive statistics
print('NaN per metric:')
print(df_metrics[TARGET_G].isna().sum())
print('\nDescriptives:')
print(df_metrics[TARGET_G].describe().round(3))

In [ ]:
df_lmm = df_metrics.dropna(subset=TARGET_G).copy()

# Binary group indicator: 0 = Control, 1 = SSD
df_lmm['group_ssd']  = (df_lmm['Groupe'] == 100).astype(float)

# Orthogonal linear contrast for difficulty level: Level 1 → -1, 2 → 0, 3 → +1
df_lmm['niveau_lin'] = df_lmm['Niveau'].map({1: -1.0, 2: 0.0, 3: 1.0})
df_lmm['grp_x_lin'] = df_lmm['group_ssd'] * df_lmm['niveau_lin']

# Z-score token count so its coefficient is on the same scale as the z-scored DVs
df_lmm['nb_tokens_z'] = (
    (df_lmm['nb_tokens'] - df_lmm['nb_tokens'].mean()) / df_lmm['nb_tokens'].std()
)

# Z-score each DV; keep raw values in *_raw columns for plotting
for col in TARGET_G:
    df_lmm[f'{col}_raw'] = df_lmm[col].copy()
    df_lmm[col] = (df_lmm[col] - df_lmm[col].mean()) / df_lmm[col].std()

df_lmm['participant'] = df_lmm['No anonyme'].astype(int).astype(str)
df_lmm['story']       = df_lmm['BD'].astype(str)

# Cross-classified variance component: one random intercept per story (BD)
vc_story = {'story': '0 + C(story)'}

print(f'LMM dataset: {len(df_lmm)} observations, '
      f'{df_lmm["participant"].nunique()} participants, '
      f'{df_lmm["story"].nunique()} stories')
print(f'  SSD    : {(df_lmm["group_ssd"]==1).sum()}')
print(f'  Control: {(df_lmm["group_ssd"]==0).sum()}')
print('\nLevel × Group balance:')
print(df_lmm.groupby(['group_ssd', 'Niveau']).size().unstack())
print('\nZ-scored metrics (μ≈0, σ≈1):')
print(df_lmm[TARGET_G].describe().round(3))

---
# Linear Mixed Models (LMM)

Procedure for each metric:
1. **Random-effects structure selection** (ML): intercept only vs. + `niveau_lin` slope
2. **Final REML model** with fixed effects (Group, linear Level, interaction, ± nb_tokens_z)
3. **Interaction LRT** χ²(1): full model vs. without interaction — **using the same random-effects structure**
4. **Residual diagnostics**: QQ-plot, residuals vs. fitted
5. **Effect sizes**: pseudo-R², Cohen’s f²

In [ ]:
# Display labels for fixed-effect terms used in printed result tables
_labels = {
    'Intercept'    : 'Intercept',
    'group_ssd'    : 'Group (SSD)',
    'niveau_lin'   : 'Level (linear)',
    'grp_x_lin'    : 'Group × Level (lin.)',
    'nb_tokens_z'  : 'N tokens (z)',
}


def fit_lmm(formula, df_m, use_rs, method='powell'):
    """Build (but do not fit) a MixedLM with:
      - random intercept per participant
      - cross-classified variance component per story (vc_story)
      - optional random slope for niveau_lin (use_rs=True)
    Call .fit(reml=...) on the returned model object."""
    kw = dict(data=df_m, groups='participant', vc_formula=vc_story)
    if use_rs:
        kw['re_formula'] = '~niveau_lin'
    return smf.mixedlm(formula, **kw)


def select_random_slopes(formula, df_m):
    """LRT (ML) comparing random-intercept vs. random-intercept + slope for
    niveau_lin. Returns True if the random slope is retained (p < .05)."""
    r_ri = fit_lmm(formula, df_m, use_rs=False).fit(reml=False, method='powell')
    try:
        r_rs = fit_lmm(formula, df_m, use_rs=True).fit(reml=False, method='powell')
        lr = -2 * (r_ri.llf - r_rs.llf)
        ldf = max(r_rs.df_modelwc - r_ri.df_modelwc, 0)
        lp = chi2_dist.sf(lr, ldf) if ldf > 0 and lr > 0 else 1.0
        use_rs = lr > 0 and lp < 0.05
        lbl = 'retained' if use_rs else 'not retained'
        print(f'  Random slope: χ²({ldf}) = {lr:.3f}, p = {lp:.4f}  →  {lbl}')
        return use_rs
    except Exception as e:
        print(f'  Random slope: failed ({e})')
        return False


def print_fe(result, dv_label):
    """Print fixed-effects table with significance stars and ICC components.
    Returns (fe, z-values, p-values) as pandas Series."""
    fe = result.fe_params
    se = result.bse_fe
    zv = result.tvalues
    pv = result.pvalues
    rows = []
    for var in fe.index:
        s = '***' if pv[var] < .001 else '**' if pv[var] < .01 else '*' if pv[var] < .05 else ''
        rows.append({'Effect': _labels.get(var, var),
                     'β': f'{fe[var]:.3f}', 'SE': f'{se[var]:.3f}',
                     'z': f'{zv[var]:.3f}', 'p': f'{pv[var]:.4f}', 'Sig.': s})
    print(f'\n=== Fixed effects: {dv_label} ===\n')
    print(pd.DataFrame(rows).to_string(index=False))

    var_part  = result.cov_re.iloc[0, 0] if result.cov_re.values.size > 0 else float('nan')
    var_story = result.vcomp[0]
    var_total = var_part + var_story + result.scale
    print(f'\nICC(participant) = {var_part / var_total:.3f}')
    print(f'ICC(story)       = {var_story / var_total:.3f}')
    return fe, zv, pv


def lrt_interaction(f_full, f_no_int, df_m, use_rs, df_lrt=1):
    """LRT χ²(1) for the interaction term.
    Uses the same random-effects structure as the final model — required for
    a valid likelihood-ratio comparison."""
    r_red = fit_lmm(f_no_int, df_m, use_rs).fit(reml=False, method='powell')
    r_ful = fit_lmm(f_full,   df_m, use_rs).fit(reml=False, method='powell')
    lr = -2 * (r_red.llf - r_ful.llf)
    lr = max(lr, 0)
    p  = chi2_dist.sf(lr, df=df_lrt)
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'n.s.'
    print(f'  LRT interaction: χ²({df_lrt}) = {lr:.3f}, p = {p:.4f}  [{sig}]')
    return lr, p


def pseudo_r2(result_full, result_null, n_obs):
    """McFadden pseudo-R² from ML log-likelihoods."""
    return 1 - (result_full.llf / result_null.llf)


def cohens_f2(r2_full, r2_reduced):
    """Cohen's f² for the incremental R² of a set of predictors."""
    if r2_full >= 1.0:
        return float('inf')
    return (r2_full - r2_reduced) / (1 - r2_full)


print('Helpers defined.')

In [6]:
all_group_rows  = []
all_inter_rows  = []
all_results     = {}

METRICS_INFO = [
    ('sg_aspl',       'ASPL'),
    ('sg_avg_degree', 'Mean Degree'),
    ('sg_lscc_size',  'LSCC size'),
    ('sg_lscc_ratio', 'LSCC ratio'),
    ('sg_tokens',     'N tokens'),
]

for dv, label in METRICS_INFO:
    print(f'\n{"=" * 70}')
    print(f'  {label} ({dv})')
    print('=' * 70)

    if dv == 'sg_tokens':
        # nb_tokens_z is the same variable as sg_tokens (z-scored),
        # so "with covariate" and "without" models are identical.
        f_full   = f'{dv} ~ group_ssd + niveau_lin + grp_x_lin'
        f_no_int = f'{dv} ~ group_ssd + niveau_lin'
        f_no_tok = f_full
    else:
        f_full   = f'{dv} ~ group_ssd + niveau_lin + grp_x_lin + nb_tokens_z'
        f_no_int = f'{dv} ~ group_ssd + niveau_lin + nb_tokens_z'
        f_no_tok = f'{dv} ~ group_ssd + niveau_lin + grp_x_lin'
    df_m     = df_lmm.dropna(subset=[dv]).copy()

    # --- 1. Random slope selection ---
    use_rs = select_random_slopes(f_full, df_m)

    # --- 2. REML final model (WITH nb_tokens_z) ---
    result = fit_lmm(f_full, df_m, use_rs).fit(reml=True, method='powell')
    fe, zv, pv = print_fe(result, f'{label} (with nb_tokens_z)')

    # --- 3. REML model WITHOUT nb_tokens_z (sensitivity) ---
    result_notok = fit_lmm(f_no_tok, df_m, use_rs).fit(reml=True, method='powell')
    fe_nt, zv_nt, pv_nt = print_fe(result_notok, f'{label} (without nb_tokens_z)')

    # --- 4. LRT interaction (consistent random-effects) ---
    lrt_chi2, lrt_p = lrt_interaction(f_full, f_no_int, df_m, use_rs, df_lrt=1)

    # --- 5. Cohen's f² for interaction ---
    r_full_ml   = fit_lmm(f_full,   df_m, use_rs).fit(reml=False, method='powell')
    r_noint_ml  = fit_lmm(f_no_int, df_m, use_rs).fit(reml=False, method='powell')
    r_null_ml   = fit_lmm(f'{dv} ~ 1', df_m, False).fit(reml=False, method='powell')

    r2_full = pseudo_r2(r_full_ml, r_null_ml, len(df_m))
    r2_noint = pseudo_r2(r_noint_ml, r_null_ml, len(df_m))
    f2 = cohens_f2(r2_full, r2_noint)
    print(f'\n  Pseudo-R²(full)     = {r2_full:.4f}')
    print(f'  Pseudo-R²(no inter) = {r2_noint:.4f}')
    print(f'  Cohen\'s f²(inter)  = {f2:.4f}  [{"small" if f2 < .15 else "medium" if f2 < .35 else "large"}]')

    # --- Store results ---
    all_results[dv] = {
        'result': result, 'result_notok': result_notok,
        'use_rs': use_rs, 'df_m': df_m,
        'f_full': f_full, 'f_no_int': f_no_int,
    }

    all_group_rows.append({
        'Metric': label, 'Model': dv,
        'β': fe['group_ssd'], 'z': zv['group_ssd'], 'p': pv['group_ssd'],
        'β_notok': fe_nt['group_ssd'], 'z_notok': zv_nt['group_ssd'], 'p_notok': pv_nt['group_ssd'],
    })

    all_inter_rows.append({
        'Metric': label, 'Model': dv,
        'β_int': fe['grp_x_lin'], 'z_wald': zv['grp_x_lin'], 'p_wald': pv['grp_x_lin'],
        'χ²_lrt': lrt_chi2, 'p_lrt': lrt_p, 'f2': f2,
        'β_int_notok': fe_nt['grp_x_lin'], 'z_notok': zv_nt['grp_x_lin'], 'p_notok': pv_nt['grp_x_lin'],
    })

print('\n' + '=' * 70)
print('Models complete.')


  ASPL (sg_aspl)


  Pente aléatoire : χ²(3) = 46.401, p = 0.0000  →  RETENUE

=== Effets fixes : ASPL (avec nb_tokens_z) ===

                 Effet      β    ES      z      p Sig.
             Intercept  0.111 0.053  2.075 0.0380    *
          Groupe (SSD) -0.261 0.081 -3.208 0.0013   **
     Niveau (linéaire) -0.037 0.047 -0.789 0.4299     
Groupe × Niveau (lin.) -0.095 0.069 -1.376 0.1688     
         Nb tokens (z) -0.307 0.034 -9.040 0.0000  ***

ICC(participant) = 0.124
ICC(histoire)    = 0.499

=== Effets fixes : ASPL (SANS nb_tokens_z) ===

                 Effet      β    ES      z      p Sig.
             Intercept  0.095 0.060  1.563 0.1180     
          Groupe (SSD) -0.224 0.092 -2.428 0.0152    *
     Niveau (linéaire) -0.146 0.045 -3.204 0.0014   **
Groupe × Niveau (lin.) -0.096 0.070 -1.381 0.1673     

ICC(participant) = 0.166
ICC(histoire)    = 0.472
  LRT interaction : χ²(1) = 1.899, p = 0.1682  [n.s.]

  Pseudo-R²(full)     = 0.0626
  Pseudo-R²(no inter) = 0.0620
  Cohen's f²(inter)

---
## Residual Diagnostics

In [ ]:
# QQ-plots assess normality of residuals; residuals-vs-fitted assess homoscedasticity.
# Shapiro-Wilk is run on a random sample of 500 obs (full sample exceeds its power range).
fig, axes = plt.subplots(len(METRICS_INFO), 2, figsize=(12, 4 * len(METRICS_INFO)))

for i, (dv, label) in enumerate(METRICS_INFO):
    result = all_results[dv]['result']
    resid  = result.resid
    fitted = result.fittedvalues

    ax_qq = axes[i, 0]
    stats.probplot(resid, dist='norm', plot=ax_qq)
    ax_qq.set_title(f'{label} — QQ plot', fontsize=13, fontweight='bold')
    ax_qq.get_lines()[0].set(markersize=3, alpha=0.5)

    ax_rf = axes[i, 1]
    ax_rf.scatter(fitted, resid, alpha=0.25, s=12, edgecolors='none')
    ax_rf.axhline(0, color='red', linewidth=0.8, linestyle='--')
    ax_rf.set_xlabel('Fitted values')
    ax_rf.set_ylabel('Residuals')
    ax_rf.set_title(f'{label} — Residuals vs Fitted', fontsize=13, fontweight='bold')

    sw_stat, sw_p = shapiro(resid.sample(min(500, len(resid)), random_state=42))
    print(f'{label}: Shapiro-Wilk W = {sw_stat:.4f}, p = {sw_p:.4f}')

plt.tight_layout()
plt.show()

---
## Summary — Multiple Comparisons Correction

Two families of tests:
- **Main effect of Group**: 3 tests (one per metric)
- **Group × Level interaction (LRT χ²(1))**: 3 tests

Correction applied: **Benjamini-Hochberg FDR** (q = 0.05); **Holm-Bonferroni** also reported.

In [ ]:
def sig_label(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''


# ── Summary 1: Group Effect ──
df_grp = pd.DataFrame(all_group_rows)

_, p_fdr_grp, _, _ = multipletests(df_grp['p'], alpha=0.05, method='fdr_bh')
_, p_holm_grp, _, _ = multipletests(df_grp['p'], alpha=0.05, method='holm')
df_grp['p_fdr']  = p_fdr_grp
df_grp['p_holm'] = p_holm_grp
df_grp['Sig']      = df_grp['p'].apply(sig_label)
df_grp['Sig_fdr']  = df_grp['p_fdr'].apply(sig_label)
df_grp['Sig_holm'] = df_grp['p_holm'].apply(sig_label)

disp = df_grp[['Metric', 'β', 'z', 'p', 'Sig', 'p_fdr', 'Sig_fdr', 'p_holm', 'Sig_holm',
               'β_notok', 'z_notok', 'p_notok']].copy()
for c in ['β', 'z', 'β_notok', 'z_notok']:
    disp[c] = disp[c].map('{:.3f}'.format)
for c in ['p', 'p_fdr', 'p_holm', 'p_notok']:
    disp[c] = disp[c].map('{:.4f}'.format)

print('=' * 110)
print('MAIN EFFECT OF GROUP (group_ssd) — z-scored metrics')
print('β_notok / p_notok columns: model WITHOUT length control')
print('=' * 110)
print(disp.to_string(index=False))

# ── Summary 2: Interaction ──
df_int = pd.DataFrame(all_inter_rows)

_, p_fdr_int, _, _ = multipletests(df_int['p_lrt'], alpha=0.05, method='fdr_bh')
_, p_holm_int, _, _ = multipletests(df_int['p_lrt'], alpha=0.05, method='holm')
df_int['p_lrt_fdr']  = p_fdr_int
df_int['p_lrt_holm'] = p_holm_int
df_int['Sig_lrt']      = df_int['p_lrt'].apply(sig_label)
df_int['Sig_lrt_fdr']  = df_int['p_lrt_fdr'].apply(sig_label)
df_int['Sig_lrt_holm'] = df_int['p_lrt_holm'].apply(sig_label)

disp_i = df_int[['Metric',
                  'β_int', 'z_wald', 'p_wald',
                  'χ²_lrt', 'p_lrt', 'Sig_lrt',
                  'p_lrt_fdr', 'Sig_lrt_fdr',
                  'p_lrt_holm', 'Sig_lrt_holm',
                  'f2']].copy()
for c in ['β_int', 'z_wald']:
    disp_i[c] = disp_i[c].map('{:.3f}'.format)
for c in ['p_wald', 'p_lrt', 'p_lrt_fdr', 'p_lrt_holm']:
    disp_i[c] = disp_i[c].map('{:.4f}'.format)
disp_i['χ²_lrt'] = disp_i['χ²_lrt'].map('{:.3f}'.format)
disp_i['f2'] = disp_i['f2'].map('{:.4f}'.format)

print('\n' + '=' * 110)
print('GROUP × LEVEL INTERACTION (lin.) — Wald (REML) + LRT χ²(1) (ML)')
print('FDR = Benjamini-Hochberg ; Holm = Holm-Bonferroni')
print('f² = Cohen\'s f² for interaction increment')
print('=' * 110)
print(disp_i.to_string(index=False))

print('\n── Sensitivity: interaction WITHOUT nb_tokens_z ──')
disp_nt = df_int[['Metric', 'β_int_notok', 'z_notok', 'p_notok']].copy()
for c in ['β_int_notok', 'z_notok']:
    disp_nt[c] = disp_nt[c].map('{:.3f}'.format)
disp_nt['p_notok'] = disp_nt['p_notok'].map('{:.4f}'.format)
print(disp_nt.to_string(index=False))

In [ ]:
def _p(p):
    """Format a p-value APA-style: no leading zero; '< .001' when p < .001."""
    if p < .001:
        return '< .001'
    return f'= {p:.3f}'.replace('= 0.', '= .')

print('=' * 80)
print('APA-STYLE RESULTS')
print('All DVs z-scored prior to analysis (M = 0, SD = 1).')
print('Fixed effects (β, z) from REML final model; LRT from ML model comparison.')
print('Interaction LRT uses the same random-effects structure as the final model.')
print('=' * 80)

print('\n── Main effect of Group (SSD vs. Control) ──\n')
for _, r in df_grp.iterrows():
    fdr  = f', FDR-corrected p {_p(r["p_fdr"])}' if r['p'] < .05 else ''
    holm = f', Holm p {_p(r["p_holm"])}' if r['p'] < .05 else ''
    nt   = f'  [without nb_tokens_z: β = {r["β_notok"]:+.2f}, p {_p(r["p_notok"])}]'
    print(f'  {r["Metric"]:12s}  β = {r["β"]:+.2f}, z = {r["z"]:+.2f}, p {_p(r["p"])}{fdr}{holm}')
    print(f'  {"":12s}{nt}')

print('\n── Group × Difficulty Level interaction (linear) ──\n')
for _, r in df_int.iterrows():
    fdr  = f', FDR-corrected p {_p(r["p_lrt_fdr"])}' if r['p_lrt'] < .05 else ''
    holm = f', Holm p {_p(r["p_lrt_holm"])}' if r['p_lrt'] < .05 else ''
    nt   = f'  [without nb_tokens_z: β = {r["β_int_notok"]:+.2f}, p {_p(r["p_notok"])}]'
    print(f'  {r["Metric"]:12s}')
    print(f'    Wald (REML) : β = {r["β_int"]:+.2f}, z = {r["z_wald"]:+.2f}, p {_p(r["p_wald"])}')
    print(f'    LRT  (ML)   : χ²(1) = {r["χ²_lrt"]:.2f}, p {_p(r["p_lrt"])}{fdr}{holm}')
    print(f'    Cohen\'s f²  = {r["f2"]:.4f}')
    print(f'    {nt}')

---
## Power Sensitivity Analysis

Estimation of the **minimum detectable β** for the interaction at 80% power, α = 0.05, based on the observed standard error of the interaction term.

In [10]:
from scipy.stats import norm

z_alpha = norm.ppf(1 - 0.05 / 2)
z_beta  = norm.ppf(0.80)

print('Post-hoc power sensitivity analysis')
print(f'α = 0.05 (two-tailed), power = 80%')
print(f'z_α/2 = {z_alpha:.3f}, z_β = {z_beta:.3f}\n')

for dv, label in METRICS_INFO:
    result = all_results[dv]['result']
    se  = result.bse_fe['grp_x_lin']
    beta_min = (z_alpha + z_beta) * se
    obs = abs(result.fe_params['grp_x_lin'])
    post_hoc_power = 1 - norm.cdf(z_alpha - obs / se)

    print(f'{label}:')
    print(f'  SE = {se:.4f}, β_min(80%) = {beta_min:.3f}, β_obs = {obs:.3f}'
          f'  {"✓ detectable" if obs >= beta_min else "✗ underpowered"}')
    print(f'  Observed post-hoc power ≈ {post_hoc_power:.1%}')
    print()

Analyse de sensibilité de puissance (post-hoc)
α = 0.05 (bilatéral), puissance = 80 %
z_α/2 = 1.960, z_β = 0.842

ASPL:
  SE = 0.0693, β_min(80%) = 0.194, β_obs = 0.095  ✗ sous-puissancé
  Puissance post-hoc observée ≈ 28.0%

Degré moyen:
  SE = 0.0351, β_min(80%) = 0.098, β_obs = 0.055  ✗ sous-puissancé
  Puissance post-hoc observée ≈ 35.2%

LSCC size:
  SE = 0.0252, β_min(80%) = 0.071, β_obs = 0.006  ✗ sous-puissancé
  Puissance post-hoc observée ≈ 4.2%

LSCC ratio:
  SE = 0.0850, β_min(80%) = 0.238, β_obs = 0.016  ✗ sous-puissancé
  Puissance post-hoc observée ≈ 3.8%

N tokens:
  SE = 0.0499, β_min(80%) = 0.140, β_obs = 0.004  ✗ sous-puissancé
  Puissance post-hoc observée ≈ 3.0%



---
## Figure — Group × Level Interaction

In [ ]:
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Times New Roman'

FONT_TITLE  = 30
FONT_LABEL  = 25
FONT_TICK   = 20
FONT_LEGEND = 20

df_plot = df_lmm.copy()
df_plot['Group'] = df_plot['group_ssd'].map({0: 'CO', 1: 'SSD'})

colors  = {'CO': '#FF8C42', 'SSD': '#E63946'}
markers = {'CO': 'o',       'SSD': 's'}

metrics_plot = [
    ('sg_aspl',       'ASPL',       'A'),
    ('sg_avg_degree', 'AD',         'B'),
    ('sg_lscc_size',  'LSCC size',  'C'),
    ('sg_lscc_ratio', 'LSCC ratio', 'D'),
    ('sg_tokens',     'N tokens',   'E'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = np.atleast_1d(axes).ravel()

for ax, (dv, lbl, panel) in zip(axes, metrics_plot):
    df_m = df_plot.dropna(subset=[dv])
    for grp in ['CO', 'SSD']:
        s = (df_m[df_m['Group'] == grp]
             .groupby('Niveau')[dv]
             .agg(['mean', 'sem'])
             .reset_index())
        ax.errorbar(
            s['Niveau'], s['mean'], yerr=s['sem'],
            marker=markers[grp], linewidth=2.2, markersize=9,
            label=grp, color=colors[grp],
            capsize=5, capthick=1.5, elinewidth=1.5,
        )

    ax.set_title(lbl, fontsize=FONT_TITLE, fontweight='bold', pad=12)
    ax.set_xlabel('referential complexity (levels)', fontsize=FONT_LABEL, labelpad=8)
    ax.set_ylabel('z-score (M ± SEM)', fontsize=FONT_LABEL, labelpad=8)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['1', '2', '3'], fontsize=FONT_TICK)
    ax.tick_params(axis='y', labelsize=FONT_TICK)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(title='Group', fontsize=FONT_LEGEND, title_fontsize=FONT_LEGEND, frameon=False)
    ax.grid(axis='y', alpha=0.25, linestyle='--', linewidth=0.8)

    ax.text(-0.12, 1.07, panel, transform=ax.transAxes,
            fontsize=20, fontweight='bold', va='top', ha='left')

# fig.suptitle(
#     'Speech Graph Metrics across Difficulty Levels by Group',
#     fontsize=FONT_TITLE + 2, fontweight='bold', y=1.02,
# )
for ax in axes[len(metrics_plot):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

---
## Cohen’s f² — Main Effect of Group

Cohen’s f² for the unique contribution of Group (group_ssd + grp_x_lin),
comparing the full model to a model without Group or interaction,
using the same random-effects structure selected above.

In [ ]:
# Cohen's f2 for the Group main effect (unique contribution over level + tokens)
# Full model    : dv ~ group_ssd + niveau_lin + grp_x_lin [+ nb_tokens_z]
# Reduced model : dv ~ niveau_lin [+ nb_tokens_z]  (removes group_ssd AND grp_x_lin)
# Same random-effects structure (use_rs) as selected above.

f2_group_rows = []

print("Cohen's f2 - Main Effect of Group\n")
print(f'{"Metric":<14}  {"R2_full":>8}  {"R2_no_group":>11}  {"f2(group)":>9}  interpretation')
print('-' * 65)

for dv, label in METRICS_INFO:
    info   = all_results[dv]
    df_m   = info['df_m']
    use_rs = info['use_rs']

    if dv == 'sg_tokens':
        f_full     = f'{dv} ~ group_ssd + niveau_lin + grp_x_lin'
        f_no_group = f'{dv} ~ niveau_lin'
    else:
        f_full     = f'{dv} ~ group_ssd + niveau_lin + grp_x_lin + nb_tokens_z'
        f_no_group = f'{dv} ~ niveau_lin + nb_tokens_z'

    r_full_ml  = fit_lmm(f_full,     df_m, use_rs).fit(reml=False, method='powell')
    r_nogrp_ml = fit_lmm(f_no_group, df_m, use_rs).fit(reml=False, method='powell')
    r_null_ml  = fit_lmm(f'{dv} ~ 1', df_m, False).fit(reml=False, method='powell')

    r2_full  = pseudo_r2(r_full_ml,  r_null_ml, len(df_m))
    r2_nogrp = pseudo_r2(r_nogrp_ml, r_null_ml, len(df_m))
    f2_g     = cohens_f2(r2_full, r2_nogrp)

    interp = 'small' if f2_g < .15 else 'medium' if f2_g < .35 else 'large'
    print(f'{label:<14}  {r2_full:>8.4f}  {r2_nogrp:>14.4f}  {f2_g:>10.4f}  {interp}')

    f2_group_rows.append({
        'Metric':      label,
        'R2_full':      r2_full,
        'R2_no_group':  r2_nogrp,
        'f2_group':     f2_g,
    })

df_f2_group = pd.DataFrame(f2_group_rows)